# Сабмит-ансамбль: ft_swa + gru_full в одном solution.py (~15 мин)
По итогам `day5_ensemble_eval` почти весь выигрыш смешивания даёт пара **ft_swa** (дообученный бейзлайн, веса эпох 2–5 усреднены) + **gru_full** (наша GRU на всех данных), с равными весами: 0.66987 на `valid_sub` против ~0.6668 у лучшей одиночной модели.

Обе модели считаются в одном вызове, скомпилированном **numba** (она есть в образе скорера: `numba==0.67.0`). Так две модели работают быстрее, чем одна наша GRU, которая уже прошла лимит по времени (≈0.5x на моих замерах; вариант через ONNX был 1.5x — не влезал в 60 минут). Матрицы весов хранятся в float16 (считается всё во float32): обе модели занимают 0.8 МБ и целиком помещаются в кэш L2 процессора скорера (1 МБ на ядро) — без этого скорость на платформе зависела бы от L3.

Перед запуском:
1. Замените на Drive папку `wunder/day2` папкой `хакатон/day2` с рабочего стола — новые файлы уже там (`build_ensemble.py`, `solution_template_ensemble_numba.py`, обновлённый `check_submission.py`).
2. **Среда выполнения → Сменить среду выполнения → T4 GPU** (GPU нужен только чтобы быстро предсказать валидацию; можно и без него, но дольше).

## 1. GPU + Drive + код + numba той же версии, что у скорера

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')
W = '/content/drive/MyDrive/wunder'
!pip -q install "numba==0.67.0" onnx
!rm -rf /content/day2 && cp -r $W/day2 /content/day2
%cd /content/day2
!python -c "import numba, numpy; print('numba', numba.__version__, '| numpy', numpy.__version__)"

## 2. Валидация — те же 600 последовательностей `valid_sub` (~1.5 мин)

In [ ]:
!python prepare_data.py --src $W/colab_data/valid_sub.parquet --out /content/data/valid

## 3. Сборка сабмита (~5 мин)
Скрипт предсказывает валидацию обеими моделями, считает коэффициенты смеси (центрирование, равные веса, масштаб до std 0.6 — ровно как в `ensemble_eval.py`), пишет `solution.py` + `weights.npz`, затем **прогоняет готовый `solution.py` построчно** на 3000 строках валидации и сверяет с torch. Если расхождение больше 1e-3, zip не создаётся.

Что должно получиться:
- `blend ...: WP on the validation mask` ≈ **0.6699** — то же число, что смесь `ft_swa+gru_full` в отчёте day5;
- `same blend with float16 weight matrices ...: WP` — то, что посчитает сабмит; отличие от строки выше должно быть в пределах ±0.0002. Если хуже больше чем на 0.0003 — пересоберите с `--weights fp32` (медленнее примерно на 20%);
- `max |diff|` порядка 1e-5.

In [ ]:
import os
FT_SWA = '+'.join(W + f'/runs/ft_baseline_full/epoch_{k}.pt' for k in (2, 3, 4, 5))
GRU_FULL = W + '/runs/gru_full/best.pt'
missing = [p for p in FT_SWA.split('+') + [GRU_FULL] if not os.path.exists(p)]
assert not missing, f'not found on Drive: {missing}'
OUT = W + '/submissions/ens_ftswa_grufull'
!python build_ensemble.py --valid-dir /content/data/valid --out $OUT --model "ft_swa=$FT_SWA@0.5" --model "gru_full=$GRU_FULL@0.5"

## 4. Проверка zip так, как его запустит платформа (~2 мин)
Распаковка в пустую папку, свежий процесс с 1 потоком, официальные правила (None на прогреве, форма (2,), конечные числа), детерминизм, сброс состояния между последовательностями и **скорость относительно нашей GRU, которая уже прошла лимит**. Нужна строка `RESULT: ready to submit` и `speed vs reference` не больше ~1.0x.

In [ ]:
REF = W + '/submissions/gru128x2_v1'          # the GRU that already passed the platform's time limit
!python check_submission.py {OUT}.zip --data $W/colab_data/valid_sub.parquet --seqs 3 --rows 20000 --ref-dir $REF

## 5. Скорость ещё раз, точнее (поочерёдные прогоны, ~1 мин)

In [ ]:
!python bench_speed.py $REF $OUT --rows 5000 --repeats 5

## 6. Скачать zip и отправить на платформу

In [ ]:
from google.colab import files
files.download(OUT + '.zip')

## Чего ждать на лидерборде
На лидерборде все наши модели выходили на ~0.011–0.015 ниже, чем на `valid_sub`. Для 0.6699 это **≈ 0.655–0.659** (сейчас лучший сабмит — 0.65083). Если время работы на платформе видно в результатах сабмита — пришлите его: по нему станет ясно, сколько запаса для третьей модели.